In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q07/annotated/checkpoints/pre_cell_5.pickle

trying: ['load_customer']
me:  4
trying: ['nation']
me:  5
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['supplier']
me:  2
trying: ['customer']
me:  4
trying: ['pd']
me:  0
trying: ['load_nation']
me:  5
trying: ['load_orders']
me:  3
trying: ['lineitem']


me:  1
trying: ['load_supplier']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['orders']


me:  3


Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable supplier
Declaring variable load_supplier
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer
Declaring variable nation
Declaring variable load_nation


In [4]:
%%RecordEvent
%%time
### cell 5 ###

# 1. Filter, compute year & volume, project only needed columns
i = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= "1995-01-01") &
        (lineitem.L_SHIPDATE <  "1997-01-01")
    ]
    .assign(
        L_YEAR=lambda df: df.L_SHIPDATE.dt.year.astype('int16'),
        VOLUME=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT)
    )[['L_ORDERKEY','L_SUPPKEY','L_YEAR','VOLUME']]
)

# 2. Pre‐filter nation table
nations = nation[
    nation.N_NAME.isin(["FRANCE","GERMANY"])
][['N_NATIONKEY','N_NAME']]

# 3. Attach nation to customer & supplier with minimal columns
cust = (
    customer[['C_CUSTKEY','C_NATIONKEY']]
    .merge(nations, left_on='C_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['C_CUSTKEY','N_NAME']]
    .rename(columns={'N_NAME':'CUST_NATION'})
)

supp = (
    supplier[['S_SUPPKEY','S_NATIONKEY']]
    .merge(nations, left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['S_SUPPKEY','N_NAME']]
    .rename(columns={'N_NAME':'SUPP_NATION'})
)

# 4. Join lineitem → orders → customer → supplier, then filter
df = (
    li
    .merge(orders[['O_ORDERKEY','O_CUSTKEY']], left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')
    .merge(cust, left_on='O_CUSTKEY', right_on='C_CUSTKEY', how='inner')
    .merge(supp, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    [['SUPP_NATION','CUST_NATION','L_YEAR','VOLUME']]
)

df = df[
    ((df.SUPP_NATION == 'FRANCE') & (df.CUST_NATION == 'GERMANY')) |
    ((df.SUPP_NATION == 'GERMANY') & (df.CUST_NATION == 'FRANCE'))
]

# 5. Aggregate on GPU and rename
total = (
    df
    .groupby(['SUPP_NATION','CUST_NATION','L_YEAR'], as_index=False)['VOLUME']
    .sum()
    .rename(columns={'VOLUME':'REVENUE'})
)

NameError: name 'li' is not defined

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/rewritten/o4_mini_high/checkpoints/post_cell_5_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q07/opt_cell_exec_info_5_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [ ]:
opt_output = Out.get(4)